# Day 11 — ILT 2: Architecture Best Practices — Cost Awareness & Environment Strategy (dev/test/prod)
### GlobalMart Data Engineering · 3:00 PM – 4:00 PM

---

## Why This Session Stands Alone

This is a standalone architecture session — the real course calendar lists it separately from the main GlobalMart Bronze → Silver → Gold pipeline build, and it stays that way here too. Nothing in this notebook depends on Day 1–10's pipeline work, and nothing later in the course depends on this notebook. The topic is how a real team thinks about **cost** and **environment separation** once a pipeline like the one you've been building is heading toward production.

## By the End of This Session You Will Be Able To

- Explain what actually drives Databricks cost: cluster uptime (DBU + cloud VM cost), job vs. all-purpose clusters, Photon/serverless, and storage vs. compute
- Apply concrete cost-awareness practices: `autotermination_minutes`, right-sizing, spot instances, and choosing job clusters over all-purpose clusters for scheduled pipelines
- Describe a real dev/test/prod environment strategy for Unity Catalog: separate catalogs, separate external locations/storage credentials, and a Git-based promotion workflow
- Look at this course's own real `gbmart` environment and explain how a production team would separate it from dev/test

---

## Before You Start — Safety Note

> **This notebook only ever reads metadata. It never creates, starts, resizes, or edits a real cluster, job, warehouse, or pipeline.** Every query below is either a `SHOW` metadata command, a read-only `SELECT` against a system table (wrapped in error handling, since it may need admin permissions this account doesn't have), or plain Python building/printing a dictionary or doing arithmetic. Nothing here provisions or bills for anything beyond the ordinary cluster you're already attached to in order to run any notebook at all.

## Part 1 — Cost Drivers in Databricks

Before you can be cost-aware, you need to know what actually generates the bill. On Azure Databricks there are three drivers, and they are **not equally sized** for most data engineering workloads.

### 1. Cluster uptime — DBU + cloud VM cost, billed by the minute
Two meters run at the same time for every minute a cluster is up, whether or not it's doing useful work:
- **DBU (Databricks Unit)** — Databricks' own usage-based fee, priced per compute type (All-Purpose Compute, Jobs Compute, SQL Compute each have a different DBU rate)
- **Cloud VM cost** — Azure bills for the underlying driver + worker virtual machines directly, on top of the DBU fee

A cluster sitting idle between two meetings costs exactly the same, minute for minute, as a cluster running a real job. This is the single biggest lever in this whole session.

### 2. Job clusters vs. all-purpose (interactive) clusters

| | All-Purpose / Interactive Cluster | Job Cluster |
|---|---|---|
| Created | Manually, or attached to for notebook work | Automatically, per job run |
| Lifetime | Stays up until manually stopped or idle-auto-terminated | Terminates automatically the instant the job finishes |
| Best for | Exploration, development, ad-hoc analysis, teaching | Scheduled / production pipelines |
| Typical risk | Left running overnight or over a weekend by accident | Essentially none — it cannot outlive the job |

### 3. Photon and Serverless

**Photon** is Databricks' native-code query engine — same cluster, often finishes work faster, which can *reduce* total cost even at a higher per-DBU rate, because you pay for less wall-clock time. **Serverless** compute (SQL warehouses, serverless jobs) removes cluster startup/idle time from the equation entirely — you pay for query execution, not for a cluster sitting around waiting for the next command.

### 4. Storage — real cost, but comparatively small

ADLS storage (GlobalMart's `ecomadlsdata` account) bills per GB/month plus transactions, and is genuinely cheap next to compute. For almost every data engineering workload, **compute dominates the bill** — storage is closer to a rounding error by comparison. That's why the rest of this session is entirely about compute.

> **Where would you actually see these numbers in a real workspace?** Two Unity Catalog system tables exist for exactly this: `system.billing.usage` (usage/DBU records) and `system.compute.clusters` (cluster inventory). Both usually require an account admin to grant access — let's try them for real against this workspace.

In [ ]:
# ─── PART 1 — Real, read-only attempt: where cost data actually lives ─────────────────
# system.billing.usage and system.compute.clusters are real Unity Catalog SYSTEM TABLES —
# a genuine live picture of spend and cluster inventory. They usually require an account
# admin to grant access to the `system` catalog, which a typical class/demo account may not
# have. We try for real; if it fails, we explain what you'd see instead of erroring out.
# Nothing below creates, starts, resizes, or edits anything — SELECT only.

try:
    billing_preview_df = spark.sql("""
        SELECT usage_date, sku_name, usage_quantity, usage_unit
        FROM system.billing.usage
        ORDER BY usage_date DESC
        LIMIT 10
    """)
    billing_preview_df.display()
    print("Access confirmed -- system.billing.usage is readable from this account.")
except Exception as e:
    print("system.billing.usage is NOT readable from this account (expected for most non-admin accounts).")
    print(f"   {type(e).__name__}: {e}")
    print()
    print("What you'd see with admin-granted access: one row per day per SKU")
    print("(e.g. PREMIUM_ALL_PURPOSE_COMPUTE, STANDARD_JOBS_COMPUTE, SERVERLESS_SQL) with the")
    print("DBUs consumed that day -- the exact raw feed a real cost dashboard is built from.")

print("-" * 88)

try:
    clusters_preview_df = spark.sql("""
        SELECT cluster_id, cluster_name, cluster_source, driver_node_type, auto_termination_minutes
        FROM system.compute.clusters
        LIMIT 10
    """)
    clusters_preview_df.display()
    print("Access confirmed -- system.compute.clusters is readable from this account.")
except Exception as e:
    print("system.compute.clusters is NOT readable from this account (also expected without admin access).")
    print(f"   {type(e).__name__}: {e}")
    print()
    print("What you'd see with admin-granted access: every cluster ever created in the workspace,")
    print("whether cluster_source was JOB or UI/API (all-purpose), and its auto_termination_minutes --")
    print("exactly how a platform team audits 'who left something running all weekend.'")

## Part 2 — Cost Awareness Practices

Knowing the drivers is step one. Here are four concrete practices that turn "we should be more cost-aware" into things you actually configure:

### 1. Set `autotermination_minutes` on every interactive cluster
An idle cluster costs the same per minute as a busy one. `autotermination_minutes` shuts it down automatically after N minutes of inactivity. There is effectively no good reason to run an interactive cluster with this set to `0` (never) outside of a very short, actively-supervised session.

### 2. Right-size the cluster to the data
A 32-core cluster reading a 10 GB table isn't "extra safe" — it's idle capacity you're paying for. Start small, watch the Spark UI for spill/shuffle pressure, and scale up only when you see evidence you need to.

### 3. Use spot (preemptible) instances for non-critical batch work
Spot/preemptible VMs cost a fraction of on-demand pricing. Azure can reclaim them on short notice, so they're a poor fit for anything latency-critical or stateful mid-computation — but a great fit for batch jobs that can simply retry (`SPOT_WITH_FALLBACK_AZURE` falls back to on-demand automatically if spot capacity isn't available).

### 4. Use job clusters — not all-purpose clusters — for scheduled/production pipelines
This is the single highest-leverage habit in this whole session. A job cluster is created fresh for one job run and destroyed the instant that run ends, successful or failed. There is no "forgot to turn it off" failure mode, because it was never left on in the first place. All-purpose clusters exist for people to attach to and explore interactively; production pipelines should never run on one.

In [ ]:
# ─── PART 2 — Cluster configs as plain data, for comparison only ──────────────────────
# These are ordinary Python dicts, built and printed for teaching purposes. NOTHING in this
# cell calls a cluster API, a WorkspaceClient, or any cluster REST endpoint -- no cluster is
# created, started, resized, or edited by running this cell.

wasteful_interactive_cluster_config = {
    "cluster_name": "shared-interactive-left-on",
    "num_workers": 8,
    "node_type_id": "Standard_DS5_v2",          # 16 cores / 56 GB per node -- oversized for a 10GB table
    "autotermination_minutes": 0,               # 0 means "never auto-terminate", NOT "instant" -- runs 24/7 if nobody stops it
    "azure_attributes": {"availability": "ON_DEMAND_AZURE"},
    "runtime_engine": "STANDARD",
}

cost_aware_interactive_cluster_config = {
    "cluster_name": "dev-interactive-rightsized",
    "num_workers": 2,
    "node_type_id": "Standard_DS3_v2",          # 4 cores / 14 GB per node -- right-sized for exploration
    "autotermination_minutes": 30,              # idles out 30 minutes after the last command
    "azure_attributes": {"availability": "SPOT_WITH_FALLBACK_AZURE"},
    "runtime_engine": "PHOTON",
}

cost_aware_job_cluster_config = {
    "cluster_name": "nightly-bronze-to-gold-job-cluster",
    "num_workers": 4,
    "node_type_id": "Standard_DS4_v2",
    # No "autotermination_minutes" key at all -- job clusters don't need one. Created for one
    # job run and torn down automatically the moment that run finishes, success or failure.
    "azure_attributes": {"availability": "SPOT_WITH_FALLBACK_AZURE"},
    "runtime_engine": "PHOTON",
}

print("WASTEFUL -- all-purpose cluster, left on:")
print(f"  {wasteful_interactive_cluster_config}")
print()
print("COST-AWARE -- right-sized interactive cluster for dev work:")
print(f"  {cost_aware_interactive_cluster_config}")
print()
print("COST-AWARE -- job cluster for a scheduled production pipeline:")
print(f"  {cost_aware_job_cluster_config}")

> **Look again at `autotermination_minutes: 0` in the wasteful config above.** `0` doesn't mean "terminates instantly" — it means "never auto-terminate." That single field, left at its most permissive value on a cluster nobody remembers exists, is one of the most common real sources of a surprise Databricks bill. The cost-aware configs fix it two different ways: the interactive cluster gets a real timeout (30 minutes), and the job cluster doesn't need the setting at all, because its whole lifecycle is scoped to one run.
>
> Let's put a number on exactly how much that difference is worth.

In [ ]:
# ─── PART 2 — A genuinely runnable cost estimate: always-on vs. job cluster ────────────
# Every rate below is an ILLUSTRATIVE PLACEHOLDER for teaching the ARITHMETIC PATTERN --
# NOT live Azure Databricks pricing. Always check the current Azure/Databricks pricing pages
# for real DBU rates in your region and tier before using this for real budgeting.

ILLUSTRATIVE_DBU_RATE_USD         = 0.55   # placeholder $ per DBU-hour, All-Purpose Compute tier
ILLUSTRATIVE_VM_RATE_USD_PER_HR   = 0.90   # placeholder underlying Azure VM $ per hour, per node
ILLUSTRATIVE_DBUS_PER_NODE_PER_HR = 2      # placeholder DBU consumption per node per hour
NUM_NODES = 4                              # 1 driver + 3 workers, illustrative cluster size

def hourly_cost(num_nodes):
    """Illustrative $/hour for a running cluster of this size: (DBU cost + VM cost) x nodes."""
    return num_nodes * (ILLUSTRATIVE_DBU_RATE_USD * ILLUSTRATIVE_DBUS_PER_NODE_PER_HR + ILLUSTRATIVE_VM_RATE_USD_PER_HR)

def monthly_cost_always_on(num_nodes, hours_per_day=24, days_per_month=30):
    """All-purpose cluster left running around the clock, whether or not it's doing useful work."""
    return hourly_cost(num_nodes) * hours_per_day * days_per_month

def monthly_cost_job_cluster(num_nodes, actual_processing_hours_per_day, days_per_month=30):
    """Job cluster: spins up per run, auto-terminates the moment the job finishes."""
    return hourly_cost(num_nodes) * actual_processing_hours_per_day * days_per_month

# GlobalMart's real nightly Bronze -> Silver -> Gold run takes about 1.5 hours of actual processing
ACTUAL_PROCESSING_HOURS_PER_DAY = 1.5

always_on_cost   = monthly_cost_always_on(NUM_NODES)
job_cluster_cost = monthly_cost_job_cluster(NUM_NODES, ACTUAL_PROCESSING_HOURS_PER_DAY)
savings          = always_on_cost - job_cluster_cost
savings_pct      = (savings / always_on_cost) * 100

print(f"Illustrative hourly rate for a {NUM_NODES}-node cluster : ${hourly_cost(NUM_NODES):,.2f}/hour")
print()
print(f"Always-on all-purpose cluster   (24h/day x 30 days)                 : ${always_on_cost:,.2f} / month")
print(f"Job cluster                     ({ACTUAL_PROCESSING_HOURS_PER_DAY}h/day actual work x 30 days) : ${job_cluster_cost:,.2f} / month")
print(f"Illustrative savings                                                : ${savings:,.2f} / month  ({savings_pct:.1f}% less)")
print()
print("These are illustrative numbers for teaching the PATTERN (usage-based cost x idle hours),")
print("not a real quote -- but the shape of the result (job clusters dramatically cheaper for")
print("anything that doesn't run 24/7) holds at real Azure Databricks pricing too.")

## Part 3 — Environment Strategy (dev/test/prod)

Every notebook you've run since Day 2 has pointed at one real, shared Unity Catalog catalog: **`gbmart`** — schemas `bronze`, `silver`, `gold`, external location `gbmart-ext-loc`, storage credential `ecomprojectscredentials`. That's been the right call for a training cohort: everyone needs to see the same real data. **A production team would not do this.** They'd insert dev and test between a developer's laptop and the catalog you've been using, which for them is production.

### Three layers of separation, not just one

| Layer | Why it matters |
|---|---|
| **Separate catalogs (or catalog + schema) per environment** | `gbmart_dev`, `gbmart_test`, `gbmart` (prod) — a bad `DROP TABLE` or a broken MERGE in dev cannot touch a byte of prod data, because it isn't pointed at prod's catalog at all |
| **Separate external locations / storage credentials per environment** | Even if dev code somehow referenced a prod path by mistake, a dev workspace's storage credential should have no grant on prod's ADLS container — the mistake fails on a permissions error, not a deleted file |
| **A promotion workflow, not hand-editing** | Code moves dev → test → prod through Git branches and CI, not by opening the prod notebook and typing changes directly into it (Part 4) |

> Some teams use a single catalog with environment-tagged schemas instead of three catalogs (e.g. `gbmart.dev_bronze` vs `gbmart.bronze`) when catalog-per-environment isn't practical for their metastore setup. The three-catalog pattern above is the cleaner isolation boundary and the one used in this example — either is a legitimate real-world choice; what matters is that dev and prod are never the same namespace.

Let's ground this in what's actually here today.

In [ ]:
# ─── PART 3 — Real, read-only: what catalogs exist in this workspace right now ────────
# A pure Unity Catalog metadata read -- zero cost, zero risk, nothing provisioned or changed.
catalogs_df = spark.sql("SHOW CATALOGS")
catalogs_df.display()

catalog_names = [row[0] for row in catalogs_df.collect()]
print(f"Catalogs visible to this account: {catalog_names}")

In [ ]:
# ─── PART 3 — Real, read-only: the schemas inside gbmart, the catalog used all course ─────
schemas_df = spark.sql("SHOW SCHEMAS IN gbmart")
schemas_df.display()

schema_names = [row[0] for row in schemas_df.collect()]
print(f"Schemas inside gbmart: {schema_names}")
print("bronze / silver / gold are the three you've built against since Day 2.")

In [ ]:
# ─── PART 3 — Environment naming strategy, shown as data only ─────────────────────────
# Illustrative dev/test/prod target design, anchored to this course's real, live prod values.
# This cell does not run any catalog/schema/external-location/credential creation statement --
# it only prints a design reference and checks names against the real SHOW CATALOGS output above.

environment_strategy = {
    "dev":  {"catalog": "gbmart_dev",  "external_location": "gbmart-dev-ext-loc",  "storage_credential": "ecomprojectscredentials_dev"},
    "test": {"catalog": "gbmart_test", "external_location": "gbmart-test-ext-loc", "storage_credential": "ecomprojectscredentials_test"},
    "prod": {"catalog": "gbmart",      "external_location": "gbmart-ext-loc",      "storage_credential": "ecomprojectscredentials"},
}

print("Illustrative dev/test/prod separation for GlobalMart -- anchored to this course's real prod values:")
print()
for env, cfg in environment_strategy.items():
    already_exists = cfg["catalog"] in catalog_names
    status = "EXISTS in this workspace right now (seen above)" if already_exists else "not present yet -- illustrative target state"
    print(f"[{env.upper():4}] catalog = {cfg['catalog']:<12} | external_location = {cfg['external_location']:<20} | credential = {cfg['storage_credential']:<32} -> {status}")

print()
print("The one catalog that IS real and live all course is 'gbmart' -- the prod-like environment.")
print("Everything built Day 2 onward has written directly to it. That's completely fine for a")
print("training cohort, but a real production team inserts dev and test catalogs between a")
print("developer's laptop and this catalog, exactly as illustrated above.")

## Part 4 — Promotion Workflow: Code Moves, Prod Doesn't Get Hand-Edited

The rule: **prod is only ever changed by a promoted, reviewed change — never by opening a prod notebook and editing it live.**

```
 dev branch (Databricks Repos)          "shows this actually
   |  edit notebook, test against        works before merging"
   |  gbmart_dev catalog
   v
 Pull Request  ------------->  CI runs automated checks
   |                            (lint, unit tests, maybe a
   |                             dry run against gbmart_test)
   v
 Merge to main
   |
   v
 Deploy to TEST  ---------->  smoke test against gbmart_test
   |                          catalog + its own external location
   v
 Promote to PROD  --------->  same reviewed code now points at
                               gbmart (prod catalog), deployed by
                               CI/CD -- not by a person typing into
                               the prod notebook directly
```

This is exactly why Databricks Repos (Day 2's Code Versioning session) matters beyond "personal backup habit" — it's the mechanism that makes dev → test → prod promotion possible at all. A notebook that only ever exists as a manually-maintained copy in the prod workspace has no reviewable diff, no CI gate, and no way to guarantee test and prod are actually running the same logic.

> **Common failure mode this prevents:** someone fixes an urgent bug directly in the prod notebook to "save time." The fix works, but it never gets backported into the dev/Git version — so the next legitimate deployment silently reverts it. Promotion-only changes make this structurally impossible.

## Key Takeaways

1. **Cluster uptime is the dominant Databricks cost driver for most DE workloads** — DBU + cloud VM cost accrue every minute a cluster is running, whether or not it's doing useful work. Storage is real but comparatively small.
2. **Job clusters beat all-purpose clusters for anything scheduled**, specifically because they cannot be left running by accident — they terminate the instant the job ends.
3. **`autotermination_minutes`, right-sizing, and spot instances are concrete, configurable habits** — not vague advice. `autotermination_minutes: 0` means "never," not "fast."
4. **Environment separation needs more than a different schema name** — separate catalogs (or catalog+schema), separate external locations/storage credentials, so a dev mistake is structurally incapable of touching prod data.
5. **Prod changes only through promotion** — Git branches + Databricks Repos + CI move code dev → test → prod. Nobody hand-edits the prod notebook.
6. **`gbmart` is this course's real prod-like environment** — everything since Day 2 has run directly against it, which is right for a training cohort and exactly what a real team would not do for production traffic.

## Self-Check

Before moving on, make sure you can answer each of these without looking back:

☐ Why does an idle all-purpose cluster cost the same as a busy one?
☐ Why does a job cluster structurally prevent the "forgot to turn it off" failure mode that an all-purpose cluster doesn't?
☐ What does `autotermination_minutes: 0` actually mean, and why is it risky to leave it there?
☐ Name two Unity Catalog objects (besides the catalog itself) that should be separate per environment, not shared.
☐ Why is a Git-based promotion workflow safer than fixing an urgent bug directly in a prod notebook?
☐ In this course's real environment, which catalog have you been treating as "prod" since Day 2?

**Discussion prompt for the room:** *If GlobalMart's real workspace only has one catalog (`gbmart`) today, what's the very first, lowest-effort step toward the dev/test/prod separation described in Part 3?*